# Markov Decision Processes (MDPs)

An MDP is defined by:
- **States** $S$: the set of all possible situations
- **Actions** $A$: the set of all possible moves
- **Transition function** $P(s' | s, a)$: probability of reaching state $s'$ from state $s$ with action $a$
- **Reward function** $R(s, a, s')$: immediate reward for a transition
- **Discount factor** $\gamma \in [0, 1)$: how much we value future rewards

In this notebook we build a **4x4 Gridworld** and solve it with:
1. **Value Iteration** (find optimal value function, then derive policy)
2. **Policy Iteration** (alternate between policy evaluation and improvement)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

## 1. Gridworld Environment

```
+----+----+----+----+
| S  |    |    |    |
+----+----+----+----+
|    |  X |    |    |
+----+----+----+----+
|    |    |    |    |
+----+----+----+----+
|    |    |    |  G |
+----+----+----+----+
```

- **S** = Start (0,0), **G** = Goal (3,3), **X** = Wall (1,1)
- Actions: Up, Down, Left, Right (deterministic)
- Reward: -1 per step, +10 at goal, -5 for hitting the wall
- Episode ends when the agent reaches the goal

In [ ]:
class GridWorld:
    """A simple 4x4 gridworld MDP."""
    
    ACTIONS = {
        0: (-1, 0),  # Up
        1: (1, 0),   # Down
        2: (0, -1),  # Left
        3: (0, 1),   # Right
    }
    ACTION_NAMES = {0: 'Up', 1: 'Down', 2: 'Left', 3: 'Right'}
    ACTION_ARROWS = {0: '\u2191', 1: '\u2193', 2: '\u2190', 3: '\u2192'}
    
    def __init__(self, rows=4, cols=4):
        self.rows = rows
        self.cols = cols
        self.n_states = rows * cols
        self.n_actions = 4
        self.goal = (3, 3)
        self.wall = (1, 1)
        self.gamma = 0.95
    
    def state_to_rc(self, s):
        return divmod(s, self.cols)
    
    def rc_to_state(self, r, c):
        return r * self.cols + c
    
    def step(self, state, action):
        """Return (next_state, reward, done) for deterministic transitions."""
        r, c = self.state_to_rc(state)
        
        # Goal is terminal
        if (r, c) == self.goal:
            return state, 0.0, True
        
        dr, dc = self.ACTIONS[action]
        nr, nc = r + dr, c + dc
        
        # Check boundaries
        if nr < 0 or nr >= self.rows or nc < 0 or nc >= self.cols:
            nr, nc = r, c  # stay in place
        
        # Check wall
        if (nr, nc) == self.wall:
            return self.rc_to_state(r, c), -5.0, False  # bounce back with penalty
        
        # Check goal
        if (nr, nc) == self.goal:
            return self.rc_to_state(nr, nc), 10.0, True
        
        return self.rc_to_state(nr, nc), -1.0, False


env = GridWorld()
print(f'Grid: {env.rows}x{env.cols}, States: {env.n_states}, Actions: {env.n_actions}')
print(f'Goal: {env.goal}, Wall: {env.wall}, Gamma: {env.gamma}')

## 2. The Bellman Equation

The **Bellman optimality equation** for the state-value function is:

$$V^*(s) = \max_a \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V^*(s') \right]$$

Since our environment is deterministic, $P(s'|s,a) = 1$ for the specific next state, simplifying to:

$$V^*(s) = \max_a \left[ R(s,a) + \gamma V^*(s') \right]$$

We solve this iteratively.

## 3. Value Iteration

In [ ]:
def value_iteration(env, theta=1e-6, max_iters=1000):
    """Find the optimal value function using value iteration."""
    V = np.zeros(env.n_states)
    history = [V.copy()]
    
    for i in range(max_iters):
        V_new = np.zeros(env.n_states)
        for s in range(env.n_states):
            r, c = env.state_to_rc(s)
            if (r, c) == env.goal:
                V_new[s] = 0  # terminal state
                continue
            
            action_values = []
            for a in range(env.n_actions):
                next_s, reward, done = env.step(s, a)
                if done:
                    action_values.append(reward)
                else:
                    action_values.append(reward + env.gamma * V[next_s])
            V_new[s] = max(action_values)
        
        delta = np.max(np.abs(V_new - V))
        V = V_new
        history.append(V.copy())
        
        if delta < theta:
            print(f'Value Iteration converged in {i+1} iterations (delta={delta:.2e})')
            break
    
    # Extract optimal policy
    policy = np.zeros(env.n_states, dtype=int)
    for s in range(env.n_states):
        action_values = []
        for a in range(env.n_actions):
            next_s, reward, done = env.step(s, a)
            if done:
                action_values.append(reward)
            else:
                action_values.append(reward + env.gamma * V[next_s])
        policy[s] = np.argmax(action_values)
    
    return V, policy, history


V_vi, policy_vi, vi_history = value_iteration(env)

## 4. Policy Iteration

In [ ]:
def policy_evaluation(env, policy, V=None, theta=1e-6, max_iters=1000):
    """Evaluate a fixed policy."""
    if V is None:
        V = np.zeros(env.n_states)
    
    for i in range(max_iters):
        V_new = np.zeros(env.n_states)
        for s in range(env.n_states):
            r, c = env.state_to_rc(s)
            if (r, c) == env.goal:
                V_new[s] = 0
                continue
            a = policy[s]
            next_s, reward, done = env.step(s, a)
            if done:
                V_new[s] = reward
            else:
                V_new[s] = reward + env.gamma * V[next_s]
        
        if np.max(np.abs(V_new - V)) < theta:
            return V_new
        V = V_new
    return V


def policy_improvement(env, V):
    """Greedily improve the policy based on value function."""
    policy = np.zeros(env.n_states, dtype=int)
    for s in range(env.n_states):
        action_values = []
        for a in range(env.n_actions):
            next_s, reward, done = env.step(s, a)
            if done:
                action_values.append(reward)
            else:
                action_values.append(reward + env.gamma * V[next_s])
        policy[s] = np.argmax(action_values)
    return policy


def policy_iteration(env, max_iters=100):
    """Full policy iteration algorithm."""
    policy = np.zeros(env.n_states, dtype=int)  # start with all-Up policy
    
    for i in range(max_iters):
        V = policy_evaluation(env, policy)
        new_policy = policy_improvement(env, V)
        
        if np.array_equal(new_policy, policy):
            print(f'Policy Iteration converged in {i+1} iterations')
            return V, new_policy
        policy = new_policy
    
    return V, policy


V_pi, policy_pi = policy_iteration(env)

## 5. Visualize Value Function as Heatmap

In [ ]:
def plot_value_function(V, env, title='Value Function'):
    """Plot the value function as a heatmap."""
    V_grid = V.reshape(env.rows, env.cols)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    im = sns.heatmap(V_grid, annot=True, fmt='.2f', cmap='YlOrRd',
                     linewidths=2, linecolor='black', ax=ax,
                     cbar_kws={'label': 'State Value'})
    
    # Mark wall and goal
    wr, wc = env.wall
    ax.add_patch(plt.Rectangle((wc, wr), 1, 1, fill=True, color='black', alpha=0.7))
    ax.text(wc + 0.5, wr + 0.5, 'WALL', ha='center', va='center', 
            color='white', fontsize=10, fontweight='bold')
    
    gr, gc = env.goal
    ax.add_patch(plt.Rectangle((gc, gr), 1, 1, fill=True, color='green', alpha=0.3))
    ax.text(gc + 0.5, gr + 0.7, 'GOAL', ha='center', va='center',
            color='green', fontsize=9, fontweight='bold')
    
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    plt.tight_layout()
    plt.show()


plot_value_function(V_vi, env, 'Optimal Value Function (Value Iteration)')
plot_value_function(V_pi, env, 'Optimal Value Function (Policy Iteration)')

## 6. Visualize Optimal Policy as Arrows

In [ ]:
def plot_policy(policy, env, title='Optimal Policy'):
    """Plot the policy as arrows on the grid."""
    fig, ax = plt.subplots(figsize=(8, 7))
    
    # Draw grid
    for r in range(env.rows + 1):
        ax.axhline(y=r, color='black', linewidth=2)
    for c in range(env.cols + 1):
        ax.axvline(x=c, color='black', linewidth=2)
    
    # Direction vectors for arrows
    arrow_dx = {0: 0, 1: 0, 2: -0.3, 3: 0.3}
    arrow_dy = {0: 0.3, 1: -0.3, 2: 0, 3: 0}
    
    for s in range(env.n_states):
        r, c = env.state_to_rc(s)
        x, y = c + 0.5, (env.rows - 1 - r) + 0.5  # flip y for display
        
        if (r, c) == env.wall:
            ax.add_patch(plt.Rectangle((c, env.rows - 1 - r), 1, 1, 
                                        fill=True, color='black', alpha=0.7))
            ax.text(x, y, 'WALL', ha='center', va='center', color='white',
                    fontsize=9, fontweight='bold')
            continue
        
        if (r, c) == env.goal:
            ax.add_patch(plt.Rectangle((c, env.rows - 1 - r), 1, 1,
                                        fill=True, color='green', alpha=0.3))
            ax.text(x, y, 'GOAL', ha='center', va='center', color='green',
                    fontsize=11, fontweight='bold')
            continue
        
        a = policy[s]
        dx, dy = arrow_dx[a], arrow_dy[a]
        ax.annotate('', xy=(x + dx, y + dy), xytext=(x - dx, y - dy),
                    arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2.5))
        ax.text(x, y - 0.35, env.ACTION_NAMES[a], ha='center', va='center',
                fontsize=7, color='gray')
    
    ax.set_xlim(0, env.cols)
    ax.set_ylim(0, env.rows)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14)
    ax.set_xticks(np.arange(env.cols) + 0.5)
    ax.set_xticklabels(range(env.cols))
    ax.set_yticks(np.arange(env.rows) + 0.5)
    ax.set_yticklabels(range(env.rows - 1, -1, -1))
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    plt.tight_layout()
    plt.show()


plot_policy(policy_vi, env, 'Optimal Policy (Value Iteration)')

## 7. Convergence Comparison

In [ ]:
# Show how value function evolves during value iteration
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
steps_to_show = [0, 1, 3, 10, 30, len(vi_history) - 1]

for ax, step_idx in zip(axes.flat, steps_to_show):
    V_grid = vi_history[step_idx].reshape(env.rows, env.cols)
    sns.heatmap(V_grid, annot=True, fmt='.1f', cmap='YlOrRd',
                linewidths=1, linecolor='gray', ax=ax, cbar=False,
                vmin=vi_history[-1].min(), vmax=vi_history[-1].max())
    ax.set_title(f'Iteration {step_idx}', fontsize=12)

fig.suptitle('Value Function Convergence During Value Iteration', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Verify Policies Match

In [ ]:
print('Value Iteration and Policy Iteration agree on the optimal policy:',
      np.array_equal(policy_vi, policy_pi))
print()
print('Max difference in value functions:', np.max(np.abs(V_vi - V_pi)))
print()
print('Optimal policy:')
for r in range(env.rows):
    row_str = ''
    for c in range(env.cols):
        s = env.rc_to_state(r, c)
        if (r, c) == env.goal:
            row_str += '  G  '
        elif (r, c) == env.wall:
            row_str += '  X  '
        else:
            row_str += f'  {env.ACTION_ARROWS[policy_vi[s]]}  '
    print(row_str)

## Summary

- **Value Iteration** applies the Bellman optimality update repeatedly until convergence, then extracts the policy.
- **Policy Iteration** alternates between evaluating the current policy and improving it greedily.
- Both converge to the same optimal policy. Policy iteration often converges in fewer outer iterations but each iteration is more expensive.
- The value function heatmap shows that states closer to the goal have higher values, and states near the wall are penalized.